In [ ]:
import math
import copy
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from mpl_toolkits.mplot3d import Axes3D

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'font.size': 10,
    'figure.dpi': 100
})

np.random.seed(42)

print("Environment configured for Solar Energy Output Prediction Lab")
print("Core algorithm: Gradient Descent for Linear Regression")
print("Application: Predicting solar panel output from irradiance data")

In [ ]:
# SOLAR FARM HOURLY DATASET
# Irradiance: incoming solar power in watts per square meter (W/m^2)
# Output: energy production in kilowatt-hours (kWh) from the full array

x_train = np.array([120, 250, 380, 520, 680, 800, 920, 1000, 850, 620, 400, 180], dtype=float)
y_train = np.array([105, 232, 358, 495, 648, 765, 882, 962, 815, 590, 380, 165], dtype=float)
hour_labels = [f"H{i+1:02d}" for i in range(len(x_train))]

print("SOLAR FARM DATASET")
print("=" * 65)
print(f"{'Hour':<8} {'Irradiance':<15} {'Output (kWh)':<15} {'Efficiency':<12} {'Period'}")
print("-" * 65)
for i in range(len(x_train)):
    eff = (y_train[i] / x_train[i]) * 100 if x_train[i] > 0 else 0
    if x_train[i] < 300:
        period = "Morning ramp-up"
    elif x_train[i] < 700:
        period = "Mid-range"
    else:
        period = "Peak production"
    print(f"{hour_labels[i]:<8} {x_train[i]:<15.0f} {y_train[i]:<15.0f} {eff:<12.1f}% {period}")
print("-" * 65)
print(f"Observations: {len(x_train)}")
print(f"Irradiance range: {x_train.min():.0f} - {x_train.max():.0f} W/m^2")
print(f"Output range: {y_train.min():.0f} - {y_train.max():.0f} kWh")
print(f"Average conversion efficiency: {np.mean(y_train/x_train)*100:.1f}%")
print(f"Correlation coefficient: {np.corrcoef(x_train, y_train)[0,1]:.6f}")

In [ ]:
# VISUALIZATION 1: Comprehensive solar data overview

fig = plt.figure(figsize=(18, 10))
gs = GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.3)

# --- Panel 1: Scatter plot of irradiance vs output ---
ax1 = fig.add_subplot(gs[0, 0])
period_colors = []
for x in x_train:
    if x < 300:
        period_colors.append('#3498db')
    elif x < 700:
        period_colors.append('#f39c12')
    else:
        period_colors.append('#e74c3c')
ax1.scatter(x_train, y_train, c=period_colors, s=150, edgecolors='black', zorder=5)
for i, hl in enumerate(hour_labels):
    ax1.annotate(hl, (x_train[i], y_train[i]), xytext=(x_train[i]+15, y_train[i]+8), fontsize=7)
ax1.set_xlabel('Solar Irradiance (W/m^2)')
ax1.set_ylabel('Energy Output (kWh)')
ax1.set_title('Irradiance vs Output', fontweight='bold')
ax1.grid(True, alpha=0.3)

# --- Panel 2: Irradiance distribution ---
ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(x_train, bins=6, color='#f39c12', edgecolor='black', alpha=0.7)
ax2.axvline(x=np.mean(x_train), color='red', linestyle='--', linewidth=2, label=f'Mean={np.mean(x_train):.0f}')
ax2.set_xlabel('Irradiance (W/m^2)')
ax2.set_ylabel('Frequency')
ax2.set_title('Irradiance Distribution', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

# --- Panel 3: Output distribution ---
ax3 = fig.add_subplot(gs[0, 2])
ax3.hist(y_train, bins=6, color='#2ecc71', edgecolor='black', alpha=0.7)
ax3.axvline(x=np.mean(y_train), color='red', linestyle='--', linewidth=2, label=f'Mean={np.mean(y_train):.0f}')
ax3.set_xlabel('Energy Output (kWh)')
ax3.set_ylabel('Frequency')
ax3.set_title('Output Distribution', fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3)

# --- Panel 4: Daily production curve ---
ax4 = fig.add_subplot(gs[1, 0])
ax4.plot(range(1, len(x_train)+1), y_train, 'g-o', linewidth=2, markersize=8, label='Actual Output')
ax4.fill_between(range(1, len(x_train)+1), y_train, alpha=0.15, color='green')
ax4.set_xlabel('Hour of Day')
ax4.set_ylabel('Energy Output (kWh)')
ax4.set_title('Daily Production Curve', fontweight='bold')
ax4.set_xticks(range(1, len(x_train)+1))
ax4.set_xticklabels(hour_labels, rotation=45, fontsize=8)
ax4.grid(True, alpha=0.3)
ax4.legend(fontsize=9)

# --- Panel 5: Efficiency over the day ---
ax5 = fig.add_subplot(gs[1, 1])
efficiencies = (y_train / x_train) * 100
ax5.plot(range(1, len(x_train)+1), efficiencies, 'b-o', linewidth=2, markersize=8)
ax5.axhline(y=np.mean(efficiencies), color='red', linestyle='--', label=f'Mean Eff={np.mean(efficiencies):.1f}%')
ax5.set_xlabel('Hour of Day')
ax5.set_ylabel('Conversion Efficiency (%)')
ax5.set_title('Efficiency Throughout Day', fontweight='bold')
ax5.set_xticks(range(1, len(x_train)+1))
ax5.set_xticklabels(hour_labels, rotation=45, fontsize=8)
ax5.grid(True, alpha=0.3)
ax5.legend(fontsize=9)

# --- Panel 6: Ranked output bar chart ---
ax6 = fig.add_subplot(gs[1, 2])
sort_idx = np.argsort(y_train)
sorted_hours = [hour_labels[i] for i in sort_idx]
sorted_output = y_train[sort_idx]
sorted_irr = x_train[sort_idx]
sorted_colors = [period_colors[i] for i in sort_idx]
ax6.barh(range(len(sorted_hours)), sorted_output, color=sorted_colors, edgecolor='black')
ax6.set_yticks(range(len(sorted_hours)))
ax6.set_yticklabels([f'{h} ({ir:.0f}W)' for h, ir in zip(sorted_hours, sorted_irr)], fontsize=8)
ax6.set_xlabel('Energy Output (kWh)')
ax6.set_title('Hours Ranked by Output', fontweight='bold')
ax6.grid(True, alpha=0.3, axis='x')

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#3498db', edgecolor='black', label='Morning ramp-up (<300 W/m^2)'),
    Patch(facecolor='#f39c12', edgecolor='black', label='Mid-range (300-700 W/m^2)'),
    Patch(facecolor='#e74c3c', edgecolor='black', label='Peak production (>700 W/m^2)'),
]
ax1.legend(handles=legend_elements, fontsize=7, loc='upper left')

plt.suptitle('Solar Farm Dataset: Comprehensive Overview', fontsize=16, fontweight='bold', y=0.98)
plt.show()

print("Top-left: Strong linear relationship between irradiance and output")
print("Top-center/right: Distributions show spread of irradiance and output values")
print("Bottom-left: Bell-shaped daily production curve peaking midday")
print("Bottom-center: Efficiency dips slightly during peak hours (thermal losses)")
print("Bottom-right: Hours ranked by output with irradiance context")

In [ ]:
def compute_cost(x, y, w, b):
    """
    Computes the squared-error cost function for linear regression.
    
    In this lab, cost measures total prediction error across all hourly
    observations. Lower cost means the model more accurately predicts
    solar energy output from irradiance data.
    
    Args:
      x (ndarray (m,)): Solar irradiance values (W/m^2) for m hours
      y (ndarray (m,)): Actual energy output (kWh) for m hours
      w (scalar): Slope parameter (kWh per W/m^2 of irradiance)
      b (scalar): Intercept parameter (baseline output at zero irradiance)
    
    Returns:
      total_cost (float): Average squared prediction error
        A cost of 500 means average error of ~31 kWh
        (sqrt(2 * 500) = 31.6). For a 10MW plant, that could
        mean committing 31 kW of backup power unnecessarily.
    """
    m = x.shape[0]
    cost_sum = 0
    for i in range(m):
        f_wb = w * x[i] + b
        cost_sum += (f_wb - y[i]) ** 2
    total_cost = (1 / (2 * m)) * cost_sum
    return total_cost


# Quick sanity check with initial parameters
initial_w = 0.5
initial_b = 50.0
initial_cost = compute_cost(x_train, y_train, initial_w, initial_b)
print(f"Initial guess: w={initial_w}, b={initial_b}")
print(f"Cost at initial guess: J = {initial_cost:.2f}")
print(f"Approximate average error: {math.sqrt(2 * initial_cost):.1f} kWh")

In [ ]:
def compute_gradient(x, y, w, b):
    """
    Computes the gradient of the cost function for linear regression.
    
    The gradient tells us how to adjust w and b to reduce prediction error.
    This is the compass that guides gradient descent toward optimal parameters.
    
    Args:
      x (ndarray (m,)): Solar irradiance values (W/m^2)
      y (ndarray (m,)): Energy output values (kWh)
      w (scalar): Current slope parameter
      b (scalar): Current intercept parameter
    
    Returns:
      dj_dw (float): Partial derivative of cost w.r.t. w
        Positive = slope too high, overestimating output at peak irradiance
        Negative = slope too low, underestimating output at peak irradiance
      dj_db (float): Partial derivative of cost w.r.t. b
        Positive = intercept too high, overestimating baseline output
        Negative = intercept too low, underestimating baseline output
    
    Key insight: The w-gradient is weighted by x[i], so prediction errors
    during high-irradiance hours (peak production) have outsized influence
    on slope adjustments. This is desirable: grid stability depends most
    on accurate peak-production forecasts.
    """
    m = x.shape[0]
    dj_dw = 0
    dj_db = 0
    
    for i in range(m):
        # Predict output using current parameters
        f_wb = w * x[i] + b
        
        # Calculate prediction error for this hour
        error = f_wb - y[i]
        
        # Accumulate gradient components
        # w-gradient weighted by irradiance: peak hours dominate slope correction
        dj_dw_i = error * x[i]
        dj_db_i = error
        
        dj_dw += dj_dw_i
        dj_db += dj_db_i
    
    dj_dw = dj_dw / m
    dj_db = dj_db / m
    
    return dj_dw, dj_db


# Test gradient at initial guess
dj_dw, dj_db = compute_gradient(x_train, y_train, initial_w, initial_b)
print(f"Gradient at w={initial_w}, b={initial_b}:")
print(f"  dj_dw = {dj_dw:.4f} (positive = slope too high, decrease w)")
print(f"  dj_db = {dj_db:.4f} (positive = intercept too high, decrease b)")
print(f"\nInterpretation: Both gradients are positive, meaning both w and b")
print(f"should be decreased to reduce prediction error.")

In [ ]:
# VISUALIZATION 2: Gradient field and per-hour contributions

fig = plt.figure(figsize=(18, 6))

# --- Panel 1: Cost vs w with gradient arrows ---
ax1 = fig.add_subplot(131)

x_mean, y_mean = np.mean(x_train), np.mean(y_train)
w_opt = np.sum((x_train - x_mean) * (y_train - y_mean)) / np.sum((x_train - x_mean)**2)
b_opt = y_mean - w_opt * x_mean
cost_opt = compute_cost(x_train, y_train, w_opt, b_opt)

w_range = np.linspace(0.3, 1.2, 200)
costs_w = [compute_cost(x_train, y_train, w, b_opt) for w in w_range]
ax1.plot(w_range, costs_w, 'b-', linewidth=2)
ax1.plot(w_opt, cost_opt, 'g*', markersize=15, zorder=5, label=f'Optimum w={w_opt:.4f}')

test_w_points = [0.5, 0.75, 1.0]
for wp in test_w_points:
    dj_dw_val, _ = compute_gradient(x_train, y_train, wp, b_opt)
    cp = compute_cost(x_train, y_train, wp, b_opt)
    scale = 0.00005
    ax1.annotate('', xy=(wp - scale*dj_dw_val, cp - scale*abs(dj_dw_val)*3),
                 xytext=(wp, cp),
                 arrowprops=dict(arrowstyle='->', color='red', lw=2))
    ax1.plot(wp, cp, 'ro', markersize=8)
    sign = '+' if dj_dw_val > 0 else ''
    ax1.annotate(f'grad={sign}{dj_dw_val:.0f}', (wp, cp), xytext=(wp+0.02, cp+200), fontsize=8, color='red')

ax1.set_xlabel('w (conversion efficiency, kWh per W/m^2)')
ax1.set_ylabel('Cost J(w, b_opt)')
ax1.set_title('Cost vs w with Gradient Arrows', fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# --- Panel 2: Quiver plot ---
ax2 = fig.add_subplot(132)

w_grid = np.linspace(0.3, 1.2, 20)
b_grid = np.linspace(-50, 150, 20)
W_q, B_q = np.meshgrid(w_grid, b_grid)

dW = np.zeros_like(W_q)
dB = np.zeros_like(B_q)
J_q = np.zeros_like(W_q)
for i in range(W_q.shape[0]):
    for j in range(W_q.shape[1]):
        dw, db = compute_gradient(x_train, y_train, W_q[i,j], B_q[i,j])
        dW[i,j] = dw
        dB[i,j] = db
        J_q[i,j] = compute_cost(x_train, y_train, W_q[i,j], B_q[i,j])

ax2.contourf(W_q, B_q, J_q, levels=20, cmap='viridis', alpha=0.6)
ax2.quiver(W_q, B_q, dW, dB, color='red', alpha=0.5, scale=5e5)
ax2.plot(w_opt, b_opt, 'g*', markersize=15, zorder=5, label='Minimum')
ax2.set_xlabel('w')
ax2.set_ylabel('b')
ax2.set_title('Gradient Field (Quiver)', fontweight='bold')
ax2.legend(fontsize=9)

# --- Panel 3: Per-hour gradient contributions ---
ax3 = fig.add_subplot(133)

contributions_w = []
contributions_b = []
for i in range(len(x_train)):
    f_wb = initial_w * x_train[i] + initial_b
    error = f_wb - y_train[i]
    contributions_w.append(error * x_train[i])
    contributions_b.append(error)

x_pos = np.arange(len(hour_labels))
width = 0.35
ax3.bar(x_pos - width/2, contributions_w, width, label='Contribution to dj_dw', color='steelblue', edgecolor='black')
ax3.bar(x_pos + width/2, contributions_b, width, label='Contribution to dj_db', color='coral', edgecolor='black')
ax3.set_xticks(x_pos)
ax3.set_xticklabels(hour_labels, rotation=45, fontsize=8)
ax3.axhline(y=0, color='black', linewidth=1)
ax3.set_ylabel('Gradient Contribution')
ax3.set_title('Per-Hour Gradient Contributions', fontweight='bold')
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3, axis='y')

plt.suptitle('Gradients: Direction, Field, and Hourly Contributions',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("Left: Cost curve with gradient arrows pointing downhill toward minimum.")
print("Center: Quiver plot. Arrows point AWAY from minimum (we subtract gradient).")
print("Right: Peak irradiance hours (H07, H08) have outsized w-gradient influence")
print("  because their errors are multiplied by large x values. This means")
print("  the algorithm prioritizes getting peak-production forecasts right.")

In [ ]:
def gradient_descent(x, y, w_in, b_in, alpha, num_iters, cost_function, gradient_function):
    """
    Performs batch gradient descent to optimize parameters w and b.
    
    The algorithm iteratively adjusts w and b by taking steps proportional
    to the negative gradient of the cost function. Like a ball rolling
    down the cost surface bowl until it settles at the minimum.
    
    Args:
      x (ndarray (m,)): Input features (solar irradiance in W/m^2)
      y (ndarray (m,)): Target values (energy output in kWh)
      w_in (scalar): Initial slope parameter
      b_in (scalar): Initial intercept parameter
      alpha (float): Learning rate (step size)
        Must be small for large-scale data (irradiance values ~100s)
      num_iters (int): Maximum optimization steps
      cost_function: Function to compute cost (monitoring)
      gradient_function: Function to compute gradients
    
    Returns:
      w (float): Optimized slope parameter
      b (float): Optimized intercept parameter
      J_history (list): Cost at each iteration
      p_history (list): [w, b] at each iteration
    """
    J_history = []
    p_history = []
    b = b_in
    w = w_in
    
    for i in range(num_iters):
        # STEP 1: Compute gradients BEFORE any updates (simultaneous rule)
        dj_dw, dj_db = gradient_function(x, y, w, b)
        
        # STEP 2: Update parameters opposite to gradient direction
        temp_w = w - alpha * dj_dw
        temp_b = b - alpha * dj_db
        w = temp_w
        b = temp_b
        
        # STEP 3: Record cost and parameters
        if i < 100000:
            J_history.append(cost_function(x, y, w, b))
            p_history.append([w, b])
        
        if i % math.ceil(num_iters / 10) == 0:
            print(f"Iteration {i:5d}: Cost {J_history[-1]:10.2e} | "
                  f"dj_dw={dj_dw:12.3f}, dj_db={dj_db:10.3f} | "
                  f"w={w:.6f}, b={b:.4f}")
    
    return w, b, J_history, p_history

In [ ]:
# RUN GRADIENT DESCENT: Optimize solar output model

print("=" * 75)
print("GRADIENT DESCENT: Solar Energy Output Model Optimization")
print("=" * 75)

w_init = 0.0
b_init = 0.0

# IMPORTANT: Alpha must be VERY small because irradiance values are large (120-1000)
# Gradient includes error * x[i], and x[i] can be ~1000, so gradients are large
# If we used alpha=0.01 (like housing data), the algorithm would diverge immediately
alpha = 0.0000005
iterations = 10000

print(f"\nInitial parameters: w={w_init}, b={b_init}")
print(f"Learning rate (alpha): {alpha} (very small due to large x scale)")
print(f"Max iterations: {iterations}")
print(f"Initial cost: {compute_cost(x_train, y_train, w_init, b_init):.2f}")
print()

w_final, b_final, J_hist, p_hist = gradient_descent(
    x_train, y_train, w_init, b_init, alpha,
    iterations, compute_cost, compute_gradient
)

print(f"\n{'=' * 75}")
print(f"OPTIMIZATION COMPLETE")
print(f"{'=' * 75}")
print(f"Final parameters: w={w_final:.6f}, b={b_final:.4f}")
print(f"Final cost: J={J_hist[-1]:.4f}")
print(f"Analytical optimal: w={w_opt:.6f}, b={b_opt:.4f}, J={cost_opt:.4f}")
print(f"\nModel equation: Output(kWh) = {w_final:.6f} * Irradiance + ({b_final:.2f})")
print(f"Interpretation: Each additional W/m^2 of irradiance produces")
print(f"  approximately {w_final*1000:.2f} Wh of energy output.")
print(f"Baseline output at zero irradiance: {b_final:.2f} kWh (system losses/parasitic)")

In [ ]:
# VISUALIZATION 3: Convergence diagnostics

fig = plt.figure(figsize=(18, 5))

# --- Panel 1: Cost vs iteration (early phase) ---
ax1 = fig.add_subplot(131)
ax1.plot(range(min(200, len(J_hist))), J_hist[:200], 'b-', linewidth=2)
ax1.set_title('Cost vs Iteration (Early Phase)', fontweight='bold')
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Cost J(w,b)')
ax1.grid(True, alpha=0.3)
ax1.axhline(y=cost_opt, color='green', linestyle='--', alpha=0.5, label=f'Optimal J={cost_opt:.2f}')
ax1.legend(fontsize=9)

# --- Panel 2: Cost vs iteration (late phase) ---
ax2 = fig.add_subplot(132)
start_idx = 2000
ax2.plot(range(start_idx, len(J_hist)), J_hist[start_idx:], 'b-', linewidth=2)
ax2.set_title('Cost vs Iteration (Late Phase)', fontweight='bold')
ax2.set_xlabel('Iteration')
ax2.set_ylabel('Cost J(w,b)')
ax2.grid(True, alpha=0.3)
ax2.axhline(y=cost_opt, color='green', linestyle='--', alpha=0.5, label=f'Optimal J={cost_opt:.2f}')
ax2.legend(fontsize=9)

# --- Panel 3: Parameter convergence ---
ax3 = fig.add_subplot(133)
w_history = [p[0] for p in p_hist]
b_history = [p[1] for p in p_hist]
ax3.plot(range(len(w_history)), w_history, 'b-', linewidth=2, label=f'w (target={w_opt:.4f})')
ax3.axhline(y=w_opt, color='blue', linestyle='--', alpha=0.4)
ax3b = ax3.twinx()
ax3b.plot(range(len(b_history)), b_history, 'r-', linewidth=2, label=f'b (target={b_opt:.2f})')
ax3b.axhline(y=b_opt, color='red', linestyle='--', alpha=0.4)
ax3.set_xlabel('Iteration')
ax3.set_ylabel('w value', color='blue')
ax3b.set_ylabel('b value', color='red')
ax3.set_title('Parameter Convergence', fontweight='bold')
ax3.grid(True, alpha=0.3)
lines1, labels1 = ax3.get_legend_handles_labels()
lines2, labels2 = ax3b.get_legend_handles_labels()
ax3.legend(lines1 + lines2, labels1 + labels2, fontsize=8)

plt.suptitle('Gradient Descent Convergence Diagnostics', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Left: Rapid cost decrease in first 200 iterations (steep gradient)")
print("Center: Gradual approach to minimum in later iterations (flat valley)")
print("Right: w (blue) and b (red) converge simultaneously to optimal values")
print(f"\nConvergence summary:")
print(f"  Cost: {J_hist[0]:.2f} -> {J_hist[-1]:.4f} ({(1-J_hist[-1]/J_hist[0])*100:.1f}% reduction)")
print(f"  Distance to analytical minimum: {abs(J_hist[-1]-cost_opt):.6f}")

In [ ]:
# VISUALIZATION 4: Gradient descent trajectory on contour plot

fig = plt.figure(figsize=(16, 6))

# --- Left: Full trajectory ---
ax1 = fig.add_subplot(121)

w_vals = np.linspace(-0.1, 1.5, 100)
b_vals = np.linspace(-30, 80, 100)
W_grid2, B_grid2 = np.meshgrid(w_vals, b_vals)
J_grid2 = np.zeros_like(W_grid2)
for i in range(W_grid2.shape[0]):
    for j in range(W_grid2.shape[1]):
        J_grid2[i,j] = compute_cost(x_train, y_train, W_grid2[i,j], B_grid2[i,j])

levels = np.linspace(0, 50000, 30)
cf = ax1.contourf(W_grid2, B_grid2, J_grid2, levels=levels, cmap='viridis', alpha=0.6)
ax1.contour(W_grid2, B_grid2, J_grid2, levels=levels, colors='black', linewidths=0.3, alpha=0.3)

traj_w = [p[0] for p in p_hist]
traj_b = [p[1] for p in p_hist]
ax1.plot(traj_w, traj_b, 'r.-', linewidth=1, markersize=2, alpha=0.5, label='GD Trajectory')
ax1.plot(traj_w[0], traj_b[0], 'yo', markersize=10, zorder=5, label='Start (0, 0)')
ax1.plot(w_opt, b_opt, 'g*', markersize=15, zorder=5, label=f'Optimum ({w_opt:.3f}, {b_opt:.2f})')

for i in range(0, len(traj_w), 1000):
    ax1.plot(traj_w[i], traj_b[i], 'r.', markersize=6)

ax1.set_xlabel('w (conversion efficiency)')
ax1.set_ylabel('b (baseline output)')
ax1.set_title('Full Gradient Descent Trajectory', fontweight='bold')
ax1.legend(fontsize=9)

# --- Right: Zoomed near minimum ---
ax2 = fig.add_subplot(122)
zoom_w_min, zoom_w_max = w_opt - 0.1, w_opt + 0.1
zoom_b_min, zoom_b_max = b_opt - 15, b_opt + 15

w_zoom = np.linspace(zoom_w_min, zoom_w_max, 80)
b_zoom = np.linspace(zoom_b_min, zoom_b_max, 80)
Wz, Bz = np.meshgrid(w_zoom, b_zoom)
Jz = np.zeros_like(Wz)
for i in range(Wz.shape[0]):
    for j in range(Wz.shape[1]):
        Jz[i,j] = compute_cost(x_train, y_train, Wz[i,j], Bz[i,j])

zoom_levels = np.linspace(0, 500, 20)
cf2 = ax2.contourf(Wz, Bz, Jz, levels=zoom_levels, cmap='viridis', alpha=0.6)
ax2.contour(Wz, Bz, Jz, levels=zoom_levels, colors='black', linewidths=0.3, alpha=0.3)

late_start = 5000
ax2.plot(traj_w[late_start:], traj_b[late_start:], 'r.-', linewidth=1.5, markersize=3, label='Late GD steps')
ax2.plot(w_opt, b_opt, 'g*', markersize=15, zorder=5, label='Optimum')
ax2.set_xlabel('w')
ax2.set_ylabel('b')
ax2.set_title('Zoomed: Final Convergence Steps', fontweight='bold')
ax2.legend(fontsize=9)

plt.suptitle('Gradient Descent on Solar Model Cost Surface', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("Left: Path from origin to optimal parameters. Large initial steps shrink near end.")
print("Right: Zoomed view shows fine-grained convergence to minimum.")
print("Step size decreases proportionally as gradient magnitude shrinks.")

In [ ]:
# VISUALIZATION 5: Model fit and residual analysis

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Left: Data with fitted line ---
ax1 = axes[0]
ax1.scatter(x_train, y_train, c=period_colors, s=150, edgecolors='black', zorder=5)
for i, hl in enumerate(hour_labels):
    ax1.annotate(hl, (x_train[i], y_train[i]), xytext=(x_train[i]+15, y_train[i]+8), fontsize=7)

x_line = np.linspace(0, 1100, 100)
y_line = w_final * x_line + b_final
ax1.plot(x_line, y_line, 'b-', linewidth=2, label=f'Model: w={w_final:.4f}, b={b_final:.2f}')

# Draw prediction errors
for i in range(len(x_train)):
    pred = w_final * x_train[i] + b_final
    error = pred - y_train[i]
    color = '#e74c3c' if error > 0 else '#2ecc71'
    ax1.plot([x_train[i], x_train[i]], [y_train[i], pred], color=color, linestyle='--', alpha=0.6, linewidth=1.5)

ax1.set_xlabel('Solar Irradiance (W/m^2)')
ax1.set_ylabel('Energy Output (kWh)')
ax1.set_title('Fitted Model with Prediction Errors', fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# --- Right: Residual plot ---
ax2 = axes[1]
predictions = w_final * x_train + b_final
residuals = y_train - predictions

ax2.scatter(predictions, residuals, c=period_colors, s=150, edgecolors='black', zorder=5)
ax2.axhline(y=0, color='black', linewidth=1)
ax2.set_xlabel('Predicted Output (kWh)')
ax2.set_ylabel('Residual (Actual - Predicted)')
ax2.set_title('Residual Plot: Model Diagnostics', fontweight='bold')
ax2.grid(True, alpha=0.3)

for i, hl in enumerate(hour_labels):
    ax2.annotate(hl, (predictions[i], residuals[i]), xytext=(predictions[i]+3, residuals[i]+0.5), fontsize=7)

plt.suptitle('Optimized Solar Output Prediction Model', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

mae = np.mean(np.abs(residuals))
rmse = np.sqrt(np.mean(residuals**2))
r_squared = 1 - np.sum(residuals**2) / np.sum((y_train - y_mean)**2)

print(f"Model performance:")
print(f"  MAE: {mae:.2f} kWh")
print(f"  RMSE: {rmse:.2f} kWh")
print(f"  Cost J: {J_hist[-1]:.4f}")
print(f"  R-squared: {r_squared:.6f}")
print(f"  Average prediction error: {mae/np.mean(y_train)*100:.2f}% of mean output")

In [ ]:
# LEARNING RATE COMPARISON

print("LEARNING RATE ANALYSIS: DATA-SCALE SENSITIVITY")
print("=" * 75)

alphas = [
    (1e-8, 'Too Small (slow convergence)'),
    (5e-7, 'Just Right (smooth convergence)'),
    (2e-6, 'Borderline (some oscillation)'),
    (1e-5, 'Too Large (divergence)'),
]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for idx, (alpha_val, desc) in enumerate(alphas):
    ax = axes[idx // 2][idx % 2]
    
    w_t, b_t, J_t, p_t = gradient_descent(
        x_train, y_train, 0.0, 0.0, alpha_val, 500, compute_cost, compute_gradient
    )
    
    if J_t[-1] < 1e15:
        ax.plot(range(len(J_t)), J_t, 'b-', linewidth=2)
        ax.axhline(y=cost_opt, color='green', linestyle='--', alpha=0.5, label=f'Optimal J={cost_opt:.2f}')
    else:
        ax.plot(range(len(J_t)), np.log10(np.array(J_t) + 1), 'r-', linewidth=2)
        ax.set_ylabel('log10(Cost)')
    
    converged = 'CONVERGED' if J_t[-1] < cost_opt * 2 else 'DIVERGED'
    color = 'green' if J_t[-1] < cost_opt * 2 else 'red'
    
    ax.set_title(f'alpha={alpha_val:.1e} ({desc})\nFinal J={J_t[-1]:.2e} [{converged}]',
                fontsize=10, fontweight='bold', color=color)
    ax.set_xlabel('Iteration')
    if idx < 2:
        ax.set_ylabel('Cost J(w,b)')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)
    
    print(f"alpha={alpha_val:.1e} ({desc}): Final J={J_t[-1]:.2e}, w={w_t:.6f}, b={b_t:.2f} [{converged}]")

plt.suptitle('Learning Rate Impact: Data Scale Matters', fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

print("\nKey lesson: The optimal alpha for this data is ~5e-7, vastly smaller")
print("than the 0.01 used for housing data. This is because irradiance values")
print("(~100s) are much larger than house sizes (1-2), producing larger gradients.")
print("\nIn production: Feature normalization (scaling inputs to 0-1 range)")
print("makes learning rates transferable across different datasets.")

In [ ]:
# VISUALIZATION 6: Divergence pathology

print("DEMONSTRATING DIVERGENCE WITH ALPHA=1e-5")
print("=" * 65)

w_div, b_div, J_div, p_div = gradient_descent(
    x_train, y_train, 0.0, 0.0, 1e-5, 15, compute_cost, compute_gradient
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Panel 1: w trajectory ---
ax1 = axes[0]
w_traj_div = [p[0] for p in p_div]
ax1.plot(range(len(w_traj_div)), w_traj_div, 'ro-', linewidth=2, markersize=6)
ax1.axhline(y=w_opt, color='green', linestyle='--', label=f'Target w={w_opt:.4f}')
ax1.set_xlabel('Iteration')
ax1.set_ylabel('w value')
ax1.set_title('w Oscillation During Divergence', fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# --- Panel 2: Cost explosion ---
ax2 = axes[1]
ax2.plot(range(len(J_div)), J_div, 'r-', linewidth=2, marker='o', markersize=4)
ax2.set_xlabel('Iteration')
ax2.set_ylabel('Cost J(w,b)')
ax2.set_title('Cost Explosion', fontweight='bold')
ax2.grid(True, alpha=0.3)

# --- Panel 3: Trajectory on contour ---
ax3 = axes[2]
ax3.contourf(W_grid2, B_grid2, J_grid2, levels=np.linspace(0, 50000, 30), cmap='viridis', alpha=0.4)
b_traj_div = [p[1] for p in p_div]
ax3.plot(w_traj_div, b_traj_div, 'r.-', linewidth=1.5, markersize=8, label='Divergence path')
ax3.plot(0, 0, 'yo', markersize=10, label='Start')
ax3.plot(w_opt, b_opt, 'g*', markersize=15, label='Target')
ax3.set_xlabel('w')
ax3.set_ylabel('b')
ax3.set_title('Spiraling Away from Minimum', fontweight='bold')
ax3.legend(fontsize=9)

plt.suptitle('Pathology of Divergence in Solar Model', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Left: w bounces between increasingly extreme positive and negative values.")
print("Center: Cost grows exponentially rather than shrinking.")
print("Right: Trajectory spirals outward, fleeing the optimum.")
print("\nGrid consequence: A diverged model might predict -5000 kWh output")
print("(impossible) or 50000 kWh (5x rated capacity). Dispatchers relying on")
print("such forecasts could trigger cascading grid failures.")

In [ ]:
# FINAL APPLICATION: Solar forecasting tool with uncertainty

print("=" * 75)
print("SOLAR ENERGY OUTPUT FORECASTING SYSTEM")
print("=" * 75)
print(f"Model: Output(kWh) = {w_final:.6f} * Irradiance + ({b_final:.2f})")
print(f"Accuracy: MAE = {mae:.1f} kWh, R-squared = {r_squared:.6f}")
print(f"Average prediction error: {mae/np.mean(y_train)*100:.2f}% of mean output")
print("=" * 75)

# Forecast for various weather scenarios
forecast_scenarios = [
    (150, 'Overcast morning (heavy cloud cover)'),
    (350, 'Partly cloudy (scattered clouds)'),
    (600, 'Mostly sunny (light haze)'),
    (850, 'Clear sky (optimal conditions)'),
    (1000, 'Peak irradiance (midday summer)'),
    (0, 'Nighttime (zero irradiance)'),
]

print(f"\n{'Weather Scenario':<45} {'Irradiance':<12} {'Forecast':<10} {'Uncertainty':<12} {'Grid Action'}")
print("-" * 100)

for irr, desc in forecast_scenarios:
    pred = w_final * irr + b_final
    # Uncertainty band based on RMSE
    lower = pred - 2 * rmse
    upper = pred + 2 * rmse
    uncertainty_str = f"[{lower:.0f}, {upper:.0f}]"
    
    if pred < 100:
        action = 'Deploy full backup'
    elif pred < 400:
        action = 'Partial backup ready'
    elif pred < 700:
        action = 'Monitor conditions'
    else:
        action = 'Export surplus'
    
    print(f"{desc:<45} {irr:<12.0f} {pred:<10.1f} {uncertainty_str:<12} {action}")

print("-" * 100)
print(f"\nUncertainty bands represent +/- 2 * RMSE = +/- {2*rmse:.1f} kWh")
print(f"This means actual output is expected within the band ~95% of the time.")

print("\n" + "=" * 75)
print("GRID INTEGRATION NOTES:")
print("=" * 75)
print("1. The uncertainty band widens for extreme irradiance values because")
print("   the linear model is less reliable outside the training data range.")
print("2. At nighttime (irradiance=0), the model predicts a small positive")
print(f"   output of {b_final:.2f} kWh, which is physically unrealistic.")
print("   This highlights a limitation: the linear model does not enforce")
print("   physical constraints (output cannot be negative or exceed rated capacity).")
print("3. For real deployment, this model should be combined with:")
print("   - Weather forecast data (cloud cover predictions)")
print("   - Temperature correction (panels lose efficiency when hot)")
print("   - Seasonal adjustment (sun angle affects effective irradiance)")
print("   - Physical constraints (clip at zero and rated capacity)")
print("4. Despite limitations, the R-squared of", f"{r_squared:.4f}", "shows irradiance alone")
print("   explains nearly all output variance, validating the linear approach.")